# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Clone repo cá nhân (fork) vào Colab và chuyển vào thư mục dự án
import os
REPO = 'https://github.com/taquangdung123/K4-Track-4-Day10-VisionLanguageModel.git'
if not os.path.exists('/content/K4-Track-4-Day10-VisionLanguageModel'):
    !git clone {REPO} /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# Mount Google Drive để sao lưu adapter LoRA / checkpoint sau mỗi lần chạy
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lab_day10'
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Kiểm tra công cụ của đề: kỳ vọng `68 passed, 1 skipped`
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Bước 1 · Kiểm tra dữ liệu: thống kê từng split, rò rỉ session, vẽ 6 ảnh có box
import json, random
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path('data')
splits = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}
meta = json.loads((DATA / 'splits' / 'meta.json').read_text(encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
assert meta['dataset_version'].startswith('d10-')

for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    n_empty = sum(not r['objects'] for r in rows)
    print(f"{s:5s} {len(rows):5d} ảnh | {sum(boxes.values()):4d} box "
          f"{dict((c, boxes[c]) for c in CLASSES)} | ảnh không box: {n_empty} | "
          f"{len({r['session'] for r in rows})} session")

# Không session nào được nằm ở hai split
sess = {s: {r['session'] for r in rows} for s, rows in splits.items()}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = sess[a] & sess[b]
    assert not overlap, f'Rò rỉ session {a}/{b}: {overlap}'
print('OK: không session nào xuất hiện ở hai split')

# Vẽ 6 ảnh train có box (nhãn YOLO chuẩn hoá [0,1] → pixel)
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
def draw(row):
    img = Image.open(DATA / row['image_path']).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    for o in row['objects']:
        xc, yc, w, h = o['yolo']
        box = ((xc - w/2) * W, (yc - h/2) * H, (xc + w/2) * W, (yc + h/2) * H)
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 2, box[1] + 2), o['label'], fill=COLORS[o['label']])
    d.text((4, 4), row['session'], fill='white')
    return img

random.seed(0)
samples = random.sample([r for r in splits['train'] if r['objects']], 6)
tiles = [draw(r) for r in samples]
W, H = tiles[0].size
grid = Image.new('RGB', (W * 3, H * 2))
for i, t in enumerate(tiles):
    grid.paste(t, ((i % 3) * W, (i // 3) * H))
Path('report/figures').mkdir(parents=True, exist_ok=True)
grid.save('report/figures/step1_train_samples.jpg', quality=85)
grid

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Bước 2 · Nhãn YOLO → JSON thang 0–1000, prompt, và kiểm tra quy đổi ngược về pixel
import json

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo  # chuẩn hoá [0, 1], tâm + kích thước
    clamp = lambda v: int(round(max(0.0, min(1000.0, v * 1000))))
    x1, y1 = clamp(xc - w / 2), clamp(yc - h / 2)
    x2, y2 = clamp(xc + w / 2), clamp(yc + h / 2)
    if x2 <= x1: x2 = min(1000, x1 + 1)
    if y2 <= y1: y2 = min(1000, y1 + 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))  # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Kiểm tra: JSON 0–1000 → pixel phải trùng box gốc (sai số làm tròn ≤ 1 phần nghìn ≈ 0.64px)
row = splits['train'][0]
ans = objects_to_answer(row['objects'])
print(row['image_id'], '→', ans)
W, H = row['width'], row['height']
max_err = 0.0
for r in splits['train']:
    for o in r['objects']:
        xc, yc, w, h = o['yolo']
        gt = [(xc - w/2) * W, (yc - h/2) * H, (xc + w/2) * W, (yc + h/2) * H]
        gt = [min(max(v, 0), lim) for v, lim in zip(gt, (W, H, W, H))]
        b = yolo_to_box(o['label'], o['yolo'])['bbox_2d']
        px = [b[0] / 1000 * W, b[1] / 1000 * H, b[2] / 1000 * W, b[3] / 1000 * H]
        max_err = max(max_err, *(abs(a - g) for a, g in zip(px, gt)))
print(f'Sai số lớn nhất khi quy ngược về pixel trên toàn train: {max_err:.2f}px')
assert max_err < 1.0

# Vẽ lại ảnh từ JSON 0–1000: phải trùng ảnh vẽ từ nhãn YOLO ở bước 1
img = Image.open(DATA / row['image_path']).convert('RGB')
d = ImageDraw.Draw(img)
for b in json.loads(ans):
    x1, y1, x2, y2 = b['bbox_2d']
    d.rectangle((x1/1000*W, y1/1000*H, x2/1000*W, y2/1000*H), outline=COLORS[b['label']], width=3)
img

## 3. E0: zero-shot

In [ ]:
# Bước 3 · E0 zero-shot: nạp Qwen3-VL-4B 4-bit (ViT + lm_head giữ fp16), suy luận test → work/e0.jsonl
import os, time, torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import write_jsonl

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['visual', 'lm_head'])  # ViT (kể cả projector) và lm_head không lượng tử hoá
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quantization, dtype=torch.float16, device_map={'': 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'  # bắt buộc khi generate theo batch
    return model, processor

def user_message(img):
    return {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}

@torch.no_grad()
def run_inference(model, processor, rows, out_path, batch_size=4, max_new_tokens=256):
    """Suy luận và ghi {"image_id", "raw"} từng dòng. Chạy lại sẽ tiếp tục từ chỗ dừng."""
    out_path = Path(out_path)
    done = {r['image_id'] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    model.eval()
    t0 = time.time()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            convs = [[user_message(Image.open(DATA / r['image_path']).convert('RGB'))] for r in batch]
            inputs = processor.apply_chat_template(
                convs, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt', padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            raws = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for r, raw in zip(batch, raws):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
            print(f'\r{len(done) + i + len(batch)}/{len(rows)} ảnh, {time.time() - t0:.0f}s', end='')
    print()

model, processor = load_base_model()
print(f'VRAM sau khi nạp: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
run_inference(model, processor, splits['test'], 'work/e0.jsonl')

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
!cp work/e0.jsonl {DRIVE_DIR}/  # sao lưu output thô lên Drive

In [ ]:
# Xem output thô của E0 để trả lời Câu 2: model trả gì, lỗi phổ biến nào?
from scripts.score import parse_output
e0 = read_jsonl('work/e0.jsonl')
bad = [r for r in e0 if parse_output(r['raw']) is None]
print(f'parse_valid_rate = {1 - len(bad) / len(e0):.3f}  ({len(bad)} output hỏng)')
print(f"Bọc ```json: {sum('```' in r['raw'] for r in e0)} · output rỗng []: {sum(r['raw'].strip() == '[]' for r in e0)}")
for r in e0[:5] + bad[:3]:
    print('---', r['image_id']); print(r['raw'][:400])

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9